Art Style Classification Pipeline
================================================
Features:
  1. Art-friendly Data Augmentation (No aggressive distortions).
  2. Single Balancing Strategy (WeightedRandomSampler only, NO double loss weighting).
  3. Stable AMP Training with robust Scheduler Step logic.
  4. 2-Stage Fine-tuning for Pretrained Models.
  5. Rich Metrics: Top-1 Acc, Top-3 Acc, Macro F1, Confusion Matrix.
  6. Soft-Voting Ensemble on Test Set.

In [1]:
# ============================================================
# 0. IMPORTS
# ============================================================
import os
import csv
import time
import random
import gc
import copy
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import cv2
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

In [2]:
# ============================================================
# 1. CONFIGURATION
# ============================================================
SOURCE_DIR  = '/kaggle/input/datasets/steubk/wikiart'
TARGET_DIR  = '/kaggle/working/modern_art_224'
OUTPUT_DIR  = '/kaggle/working/outputs'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

TARGET_CLASSES = [
    'Pop_Art', 'Minimalism', 'Symbolism', 'Impressionism',
    'Fauvism', 'Naive_Art_Primitivism', 'Cubism', 'Abstract_Expressionism',
    'Expressionism', 'Post_Impressionism', 'Art_Nouveau_Modern',
    'Color_Field_Painting', 'Action_painting', 'New_Realism',
    'Synthetic_Cubism', 'Analytical_Cubism', 'Contemporary_Realism',
]

MAX_IMAGES_PER_CLASS = 1200
IMG_SIZE             = 224
BATCH_SIZE           = 64
NUM_CLASSES          = len(TARGET_CLASSES)

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_gpus = torch.cuda.device_count()

In [3]:
# ============================================================
# 2. DATASET PREPARATION & PIPELINE
# ============================================================
def _process_image(args: tuple) -> bool:
    src, dst = args
    try:
        img = cv2.imread(src)
        if img is None: return False
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        cv2.imwrite(dst, img)
        return True
    except: return False

def prepare_dataset(seed: int = 42):
    Path(TARGET_DIR).mkdir(exist_ok=True)
    rng = random.Random(seed)
    tasks = []
    print("--- Preparing Dataset ---")
    for style in TARGET_CLASSES:
        src_dir, dst_dir = os.path.join(SOURCE_DIR, style), os.path.join(TARGET_DIR, style)
        if not os.path.isdir(src_dir): continue
        Path(dst_dir).mkdir(exist_ok=True)
        all_imgs = [f for f in os.listdir(src_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        selected = rng.sample(all_imgs, min(MAX_IMAGES_PER_CLASS, len(all_imgs)))
        for name in selected:
            tasks.append((os.path.join(src_dir, name), os.path.join(dst_dir, name)))
            
    with ProcessPoolExecutor(max_workers=4) as ex:
        list(tqdm(ex.map(_process_image, tasks), total=len(tasks)))

In [4]:
# ============================================================
# 3. DATA PIPELINE
# ============================================================
def build_transforms():
    """Art-friendly Augmentation: Giảm tối đa các phép biến dạng làm hỏng brush stroke/color."""
    train_tf = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        # ColorJitter RẤT NHẸ để không đổi hẳn style của tranh
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    val_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    return train_tf, val_tf

def build_dataloaders(seed: int = 42):
    train_tf, val_tf = build_transforms()
    base_ds = datasets.ImageFolder(TARGET_DIR)
    targets = base_ds.targets

    train_idx, temp_idx = train_test_split(range(len(base_ds)), test_size=0.2, stratify=targets, random_state=seed)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, stratify=[targets[i] for i in temp_idx], random_state=seed)

    train_ds = Subset(datasets.ImageFolder(TARGET_DIR, transform=train_tf), train_idx)
    val_ds   = Subset(datasets.ImageFolder(TARGET_DIR, transform=val_tf), val_idx)
    test_ds  = Subset(datasets.ImageFolder(TARGET_DIR, transform=val_tf), test_idx)

    # Lựa chọn duy nhất để balance: WeightedRandomSampler (KHÔNG pass weights vào CE Loss nữa)
    class_counts = np.bincount([targets[i] for i in train_idx], minlength=NUM_CLASSES)
    class_weights = 1.0 / (class_counts + 1e-6)
    sample_weights = torch.tensor([class_weights[targets[i]] for i in train_idx], dtype=torch.float)
    sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

    print(f"Dataset Split -> Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader

In [5]:
# ============================================================
# 4. MODEL DEFINITIONS
# ============================================================

class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=None, act=nn.GELU):
        super().__init__()
        if p is None:
            p = k // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            act()
        )

    def forward(self, x):
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(self, channel, reduction=16):
        super().__init__()
        hidden = max(channel // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).flatten(1)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y


class ResidualSEBlock(nn.Module):
    """
    Residual block + SE.
    Regular enough for small dataset, but still strong for texture/style.
    """
    def __init__(self, in_ch, out_ch, stride=1, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.se    = SEBlock(out_ch)
        self.drop  = nn.Dropout2d(dropout) if dropout > 0 else nn.Identity()

        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out = out + self.shortcut(x)
        return F.relu(out, inplace=True)


def make_stage(in_ch, out_ch, num_blocks, stride=2, dropout=0.0):
    layers = [ResidualSEBlock(in_ch, out_ch, stride=stride, dropout=dropout)]
    for _ in range(num_blocks - 1):
        layers.append(ResidualSEBlock(out_ch, out_ch, stride=1, dropout=dropout))
    return nn.Sequential(*layers)


# ---------- Custom Model 1: ArtResNet ----------
class ArtResNet(nn.Module):
    """
    CNN-centric model for art style:
    - local texture / brush stroke / color patterns
    - multi-scale pooling (avg + max)
    - residual + SE
    """
    def __init__(self, num_classes, dropout_fc=0.30):
        super().__init__()

        # Stem: keep enough low-level detail, but reduce early noise
        self.stem = nn.Sequential(
            ConvBNAct(3, 64, 3, 2),   # 224 -> 112
            ConvBNAct(64, 64, 3, 1),
            ConvBNAct(64, 128, 3, 1),
        )

        # 4 stages, moderate depth
        self.stage1 = make_stage(128, 128, num_blocks=2, stride=1, dropout=0.03)
        self.stage2 = make_stage(128, 256, num_blocks=2, stride=2, dropout=0.06)  # 112 -> 56
        self.stage3 = make_stage(256, 384, num_blocks=2, stride=2, dropout=0.10)  # 56 -> 28
        self.stage4 = make_stage(384, 512, num_blocks=2, stride=2, dropout=0.14)  # 28 -> 14

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.gmp = nn.AdaptiveMaxPool2d(1)

        self.classifier = nn.Sequential(
            nn.LayerNorm(512 * 2),
            nn.Dropout(dropout_fc),
            nn.Linear(512 * 2, 512),
            nn.GELU(),
            nn.Dropout(dropout_fc / 2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)

        avg = self.gap(x).flatten(1)
        mx  = self.gmp(x).flatten(1)
        feat = torch.cat([avg, mx], dim=1)
        return self.classifier(feat)


# ---------- Custom Model 2: ArtConvGRU ----------
class ArtConvGRU(nn.Module):
    """
    Hybrid model:
    - CNN stem compresses image to 7x7 feature map
    - 49 tokens only -> GRU stable hơn patch sequence dài
    - BiGRU + attention pooling
    """
    def __init__(self, num_classes, hidden_dim=256, num_layers=2, dropout=0.20):
        super().__init__()

        self.stem = nn.Sequential(
            ConvBNAct(3, 64, 3, 2),      # 224 -> 112
            ConvBNAct(64, 128, 3, 2),    # 112 -> 56
            ResidualSEBlock(128, 192, stride=2, dropout=0.03),  # 56 -> 28
            ResidualSEBlock(192, 256, stride=2, dropout=0.05),   # 28 -> 14
            nn.AdaptiveAvgPool2d((7, 7))                         # 14 -> 7x7
        )

        self.embed_dim = 256
        self.num_tokens = 7 * 7  # 49
        self.proj = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Dropout(dropout / 2),
        )
        self.pos_embed = nn.Embedding(self.num_tokens, self.embed_dim)

        self.gru = nn.GRU(
            input_size=self.embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        gru_out = hidden_dim * 2
        self.attn = nn.Sequential(
            nn.Linear(gru_out, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(gru_out),
            nn.Dropout(dropout),
            nn.Linear(gru_out, 256),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        feat = self.stem(x)                  # B, 256, 7, 7
        b, c, h, w = feat.shape

        seq = feat.flatten(2).permute(0, 2, 1)   # B, 49, 256
        pos = torch.arange(self.num_tokens, device=x.device)
        seq = self.proj(seq + self.pos_embed(pos).unsqueeze(0))

        out, _ = self.gru(seq)                    # B, 49, 512
        attn_w = F.softmax(self.attn(out), dim=1) # B, 49, 1
        pooled = (out * attn_w).sum(dim=1)        # B, 512

        return self.classifier(pooled)


# ---------- Pretrained wrappers ----------
def get_vit_model(num_classes: int) -> nn.Module:
    model = models.vit_b_16(weights='IMAGENET1K_V1')
    for p in model.parameters():
        p.requires_grad = False

    in_feat = model.heads.head.in_features
    model.heads = nn.Sequential(
        nn.LayerNorm(in_feat),
        nn.Dropout(0.20),
        nn.Linear(in_feat, num_classes)
    )

    for p in model.heads.parameters():
        p.requires_grad = True
    return model


def get_resnet50_model(num_classes: int) -> nn.Module:
    model = models.resnet50(weights='IMAGENET1K_V2')
    for p in model.parameters():
        p.requires_grad = False

    in_feat = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.20),
        nn.Linear(in_feat, num_classes)
    )

    for p in model.fc.parameters():
        p.requires_grad = True
    return model

In [6]:
# ============================================================
# 5. CSV HISTORY LOGGER
# ============================================================
class CSVLogger:
    def __init__(self, path: str):
        self.path = path
        with open(self.path, 'w', newline='') as f:
            csv.writer(f).writerow(['model', 'epoch', 'loss', 'val_acc', 'val_f1', 'lr', 'time'])

    def log(self, *args):
        with open(self.path, 'a', newline='') as f:
            csv.writer(f).writerow(args)

In [7]:
# ============================================================
# 6. TRAINING LOOP
# ============================================================
def train_model(
    model,
    name,
    total_epochs,
    train_loader,
    val_loader,
    logger,
    ce_weights=None,
    max_lr=1e-3,
    is_pretrained=False,
    warmup_epochs=3,
    patience=15,
):
    """
    Training design:
    - Custom models: train full network from scratch
    - Pretrained models: head-only warmup, then unfreeze all
    - Use ONLY one imbalance strategy here: class-weighted CE
      (do NOT combine with WeightedRandomSampler in the dataloader)
    - Step scheduler per epoch to avoid scheduler/AMP issues
    """
    print(f"\n[{name}] Bắt đầu Train | Epochs: {total_epochs} | Base LR: {max_lr}")

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    if ce_weights is not None:
        ce_weights = ce_weights.to(device)

    criterion = nn.CrossEntropyLoss(weight=ce_weights, label_smoothing=0.05)

    inner = model.module if isinstance(model, nn.DataParallel) else model

    # optimizer chỉ lấy params đang train
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=max_lr if not is_pretrained else max_lr,
        weight_decay=1e-4
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=total_epochs
    )

    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

    best_val_f1 = -1.0
    best_weights = copy.deepcopy(inner.state_dict())
    no_improve = 0

    for epoch in range(total_epochs):

        # ---- warmup stage for pretrained models ----
        if is_pretrained and epoch == warmup_epochs:
            print("  -> [Unfreeze Backbone] bắt đầu fine-tune toàn bộ mạng.")
            for p in inner.parameters():
                p.requires_grad = True

            # LR nhỏ hơn sau khi unfreeze
            optimizer = optim.AdamW(
                inner.parameters(),
                lr=max_lr * 0.2,
                weight_decay=1e-4
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=max(total_epochs - warmup_epochs, 1)
            )

        # ---- train ----
        model.train()
        running_loss = 0.0
        t0 = time.time()

        for inputs, labels in train_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels  = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                logits = model(inputs)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(inner.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()

        scheduler.step()

        # ---- validation ----
        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                    logits = model(inputs)
                preds = logits.argmax(dim=1).cpu().numpy()

                all_preds.extend(preds.tolist())
                all_labels.extend(labels.numpy().tolist())

        avg_loss = running_loss / len(train_loader)
        val_acc = 100.0 * (np.array(all_preds) == np.array(all_labels)).mean()
        val_f1  = 100.0 * f1_score(all_labels, all_preds, average='macro')
        cur_lr  = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0

        logger.log(name, epoch + 1, f"{avg_loss:.4f}", f"{val_acc:.2f}", f"{val_f1:.2f}", f"{cur_lr:.2e}", f"{elapsed:.1f}")

        flag = ""
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_weights = copy.deepcopy(inner.state_dict())
            no_improve = 0
            flag = " ★"
            torch.save(best_weights, os.path.join(OUTPUT_DIR, f"best_{name}.pth"))
        else:
            no_improve += 1

        print(
            f"  Ep {epoch+1:3d}/{total_epochs} | Loss: {avg_loss:.4f} | "
            f"Acc: {val_acc:.1f}% | MacroF1: {val_f1:.1f}% | LR: {cur_lr:.1e}{flag}"
        )

        if no_improve >= patience:
            print(f"  -> Early stopping tại epoch {epoch+1}")
            break

    inner.load_state_dict(best_weights)
    print(f"[{name}] Hoàn thành. Best Macro F1: {best_val_f1:.2f}%\n")
    return inner

In [8]:
# ============================================================
# 7. EVALUATION & ENSEMBLE ENGINE
# ============================================================
def evaluate_on_test(model, name, test_loader):
    """Chạy inference trên tập Test, trả về logits và metrics."""
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            logits = model(inputs.to(device))
            all_logits.append(logits.cpu())
            all_labels.append(labels)
            
    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    
    # Calculate Metrics
    preds = all_logits.argmax(dim=1)
    acc = 100 * (preds == all_labels).float().mean().item()
    f1  = f1_score(all_labels.numpy(), preds.numpy(), average='macro') * 100
    
    _, top3_pred = all_logits.topk(3, 1, True, True)
    top3_acc = 100 * (top3_pred == all_labels.view(-1, 1).expand_as(top3_pred)).float().sum().item() / len(all_labels)
    
    print(f"  [Test] {name:20s} | Acc: {acc:.2f}% | Top-3: {top3_acc:.2f}% | F1: {f1:.2f}%")
    return all_logits, all_labels.numpy(), preds.numpy()

def plot_confusion_matrix(y_true, y_pred, title, save_name):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES)
    plt.title(title, fontsize=15)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
    plt.close()

def ensemble_predictions(models_logits_dict, true_labels):
    """Ensemble theo phương pháp Average Logits (Soft Voting)."""
    print("\n--- Thực hiện Ensemble (Average Logits) ---")
    stacked_logits = torch.stack(list(models_logits_dict.values())) # Shape: [num_models, num_samples, num_classes]
    ensemble_logits = stacked_logits.mean(dim=0)
    ensemble_preds = ensemble_logits.argmax(dim=1).numpy()
    
    acc = 100 * (ensemble_preds == true_labels).mean()
    f1  = f1_score(true_labels, ensemble_preds, average='macro') * 100
    _, top3_pred = ensemble_logits.topk(3, 1, True, True)
    top3_acc = 100 * (top3_pred == torch.tensor(true_labels).view(-1, 1).expand_as(top3_pred)).float().sum().item() / len(true_labels)
    
    print(f"  [Ensemble Final]       | Acc: {acc:.2f}% | Top-3: {top3_acc:.2f}% | F1: {f1:.2f}%")
    plot_confusion_matrix(true_labels, ensemble_preds, 'Ensemble Confusion Matrix', 'cm_ensemble.png')
    return ensemble_preds

In [9]:
# ============================================================
# 8. MAIN — RUN PIPELINE
# ============================================================
def main():
    if not os.path.isdir(TARGET_DIR) or not os.listdir(TARGET_DIR):
        prepare_dataset()

    train_loader, val_loader, test_loader = build_dataloaders()
    logger = CSVLogger(os.path.join(OUTPUT_DIR, 'training_log.csv'))

    models_cfg = [
        # Custom models: train dài hơn, LR cao hơn (train from scratch)
        dict(name='ArtResNet', model=ArtResNet(NUM_CLASSES), epochs=100, lr=1e-3, pre=False, warm=0),
        dict(name='ArtConvGRU', model=ArtConvGRU(NUM_CLASSES), epochs=100, lr=1e-3, pre=False, warm=0),
        # Pretrained models: train ngắn hơn, LR cực nhỏ, warmup (freeze backbone)
        dict(name='ViT_Pretrained', model=get_vit_model(NUM_CLASSES), epochs=30, lr=5e-5, pre=True, warm=3),
        dict(name='ResNet50_Pretrained', model=get_resnet50_model(NUM_CLASSES), epochs=30, lr=1e-4, pre=True, warm=3),
    ]

    trained_models = {}
    
    # 1. Huấn luyện 4 models
    for cfg in models_cfg:
        trained_model = train_model(
            model=cfg['model'], name=cfg['name'], total_epochs=cfg['epochs'],
            train_loader=train_loader, val_loader=val_loader, logger=logger,
            max_lr=cfg['lr'], is_pretrained=cfg['pre'], warmup_epochs=cfg['warm']
        )
        trained_models[cfg['name']] = trained_model
        torch.cuda.empty_cache()
        gc.collect()

    # 2. Đánh giá trên tập Test & Lưu Logits
    print("\n=== KẾT QUẢ TRÊN TẬP TEST ===")
    test_logits_dict = {}
    true_labels = None
    
    for name, model in trained_models.items():
        logits, labels, preds = evaluate_on_test(model, name, test_loader)
        test_logits_dict[name] = logits
        if true_labels is None: true_labels = labels
        plot_confusion_matrix(labels, preds, f'{name} Confusion Matrix', f'cm_{name}.png')

    # 3. Chạy Ensemble (Soft Voting)
    ensemble_predictions(test_logits_dict, true_labels)

if __name__ == '__main__':
    main()

--- Preparing Dataset ---


100%|██████████| 15353/15353 [03:08<00:00, 81.65it/s]


Dataset Split -> Train: 12282 | Val: 1535 | Test: 1536
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 201MB/s]


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 184MB/s]



[ArtResNet] Bắt đầu Train | Epochs: 100 | Base LR: 0.001
  Ep   1/100 | Loss: 2.6015 | Acc: 17.5% | MacroF1: 14.4% | LR: 1.0e-03 ★
  Ep   2/100 | Loss: 2.3902 | Acc: 24.1% | MacroF1: 20.4% | LR: 1.0e-03 ★
  Ep   3/100 | Loss: 2.2944 | Acc: 24.3% | MacroF1: 19.8% | LR: 1.0e-03
  Ep   4/100 | Loss: 2.2472 | Acc: 26.8% | MacroF1: 23.2% | LR: 1.0e-03 ★
  Ep   5/100 | Loss: 2.1636 | Acc: 25.3% | MacroF1: 23.0% | LR: 9.9e-04
  Ep   6/100 | Loss: 2.1044 | Acc: 28.7% | MacroF1: 26.0% | LR: 9.9e-04 ★
  Ep   7/100 | Loss: 2.0399 | Acc: 26.6% | MacroF1: 23.8% | LR: 9.9e-04
  Ep   8/100 | Loss: 2.0050 | Acc: 30.7% | MacroF1: 26.9% | LR: 9.8e-04 ★
  Ep   9/100 | Loss: 1.9795 | Acc: 30.2% | MacroF1: 26.4% | LR: 9.8e-04
  Ep  10/100 | Loss: 1.9013 | Acc: 28.2% | MacroF1: 24.6% | LR: 9.8e-04
  Ep  11/100 | Loss: 1.8906 | Acc: 33.3% | MacroF1: 30.6% | LR: 9.7e-04 ★
  Ep  12/100 | Loss: 1.8520 | Acc: 32.6% | MacroF1: 28.6% | LR: 9.6e-04
  Ep  13/100 | Loss: 1.8125 | Acc: 35.2% | MacroF1: 32.4% | LR: 9.